# Population Work 2: Tax Contribution by Race Over Time

This notebook estimates **tax contribution by race over time** using ACS 5-year data and builds a stacked filled line chart.

Method summary:
- Primary source: ACS aggregate household income by race (`B19025*`).
- Fallback source: if aggregate income is missing, estimate as `median income (B19013*) * households (B11001*)`.
- Tax proxy: estimated effective rate applied to aggregate income.


In [1]:
import os
import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

try:
    from census import Census
except Exception:
    Census = None

pd.set_option('display.max_columns', None)
pio.renderers.default = "vscode" if os.environ.get("VSCODE_PID") else "notebook_connected"


In [2]:
# Config
CENSUS_API_KEY = "942e0a44c121ca03ced84b727df9b004f1f1367d"
YEARS = list(range(2009, 2026))

# Primary + fallback variable mapping
RACE_VARS = {
    'White':    {'agg': 'B19025A_001E', 'median': 'B19013A_001E', 'hh': 'B11001A_001E'},
    'Black':    {'agg': 'B19025B_001E', 'median': 'B19013B_001E', 'hh': 'B11001B_001E'},
    'Asian':    {'agg': 'B19025D_001E', 'median': 'B19013D_001E', 'hh': 'B11001D_001E'},
    'Hispanic': {'agg': 'B19025I_001E', 'median': 'B19013I_001E', 'hh': 'B11001I_001E'},
}

STYLE = {
    'background': '#f5efe2',
    'grid': '#d8cfbf',
    'font': '#2f2a25',
    'colors': {
        'White': '#7aa6c2',
        'Black': '#d98f75',
        'Asian': '#8bb89a',
        'Hispanic': '#e3bf78',
    }
}

census_client = Census(CENSUS_API_KEY) if (Census is not None and CENSUS_API_KEY) else None


In [3]:
def _to_numeric(df, cols):
    existing = [c for c in cols if c in df.columns]
    if existing:
        df[existing] = df[existing].apply(pd.to_numeric, errors='coerce')
    return df


def fetch_with_census_package(year, fields):
    if census_client is None:
        return None
    try:
        data = census_client.acs5.get(fields, {'for': 'state:*'}, year=year)
        return pd.DataFrame(data)
    except Exception:
        return None


def fetch_with_rest_api(year, fields, api_key):
    base = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {
        'get': ','.join(fields),
        'for': 'state:*',
    }
    if api_key:
        params['key'] = api_key

    try:
        r = requests.get(base, params=params, timeout=30)
        if r.status_code != 200:
            return None
        payload = r.json()
        if not payload or len(payload) < 2:
            return None
        df = pd.DataFrame(payload[1:], columns=payload[0])
        return df
    except Exception:
        return None


def fetch_year_data(year):
    fields = ['NAME']
    for race_map in RACE_VARS.values():
        fields.extend([race_map['agg'], race_map['median'], race_map['hh']])

    # First path: census package
    df = fetch_with_census_package(year, fields)

    # Second path: direct Census REST API
    if df is None or df.empty:
        df = fetch_with_rest_api(year, fields, CENSUS_API_KEY)

    if df is None or df.empty:
        return None

    numeric_cols = [f for f in fields if f != 'NAME']
    df = _to_numeric(df, numeric_cols)

    row = {'Year': year}

    for race, vars_map in RACE_VARS.items():
        agg_var = vars_map['agg']
        median_var = vars_map['median']
        hh_var = vars_map['hh']

        agg_total = df[agg_var].sum(skipna=True) if agg_var in df.columns else np.nan
        if pd.isna(agg_total) or agg_total == 0:
            median_sum = df[median_var].sum(skipna=True) if median_var in df.columns else np.nan
            hh_sum = df[hh_var].sum(skipna=True) if hh_var in df.columns else np.nan
            if pd.notna(median_sum) and pd.notna(hh_sum):
                agg_total = median_sum * hh_sum

        row[f'{race}_AggregateIncome'] = float(agg_total) if pd.notna(agg_total) else np.nan

        # Dynamic effective-rate proxy from median income level (bounded)
        median_mean = df[median_var].mean(skipna=True) if median_var in df.columns else np.nan
        if pd.notna(median_mean):
            eff_rate = np.clip(0.08 + (median_mean / 100000.0) * 0.15, 0.08, 0.28)
        else:
            eff_rate = 0.18

        row[f'{race}_EffRate'] = float(eff_rate)
        row[f'{race}_TaxContribution'] = row[f'{race}_AggregateIncome'] * eff_rate if pd.notna(row[f'{race}_AggregateIncome']) else np.nan

    return row


In [4]:
# Build time series across available years
records = []
for y in YEARS:
    rec = fetch_year_data(y)
    if rec is not None:
        records.append(rec)

if not records:
    # Local fallback if API is not reachable
    fallback_csv = 'output/tax_contribution_by_race_over_time.csv'
    if os.path.exists(fallback_csv):
        df_tax = pd.read_csv(fallback_csv)
        print(f'Loaded local fallback: {fallback_csv}')
    else:
        raise RuntimeError('No data fetched and no local fallback CSV found.')
else:
    df_tax = pd.DataFrame(records).sort_values('Year').reset_index(drop=True)
    os.makedirs('output', exist_ok=True)
    df_tax.to_csv('output/tax_contribution_by_race_over_time.csv', index=False)
    print('Saved output/tax_contribution_by_race_over_time.csv')

df_tax.head()


Saved output/tax_contribution_by_race_over_time.csv


,Year,White_AggregateIncome,White_EffRate,White_TaxContribution,Black_AggregateIncome,Black_EffRate,Black_TaxContribution,Asian_AggregateIncome,Asian_EffRate,Asian_TaxContribution,Hispanic_AggregateIncome,Hispanic_EffRate,Hispanic_TaxContribution
0,2009,6.560237e+12,0.161444,1.059109e+12,6.225591e+11,0.131813,8.206165e+10,3.683627e+11,0.168404,6.203371e+10,6.889502e+11,0.139823,9.633130e+10
1,2010,6.684937e+12,0.162402,1.085647e+12,6.555950e+11,0.132861,8.710332e+10,4.036935e+11,0.168533,6.803557e+10,7.361838e+11,0.140192,1.032074e+11
2,2011,6.872856e+12,0.164262,1.128948e+12,6.768300e+11,0.133171,9.013402e+10,4.238765e+11,0.171526,7.270590e+10,7.678761e+11,0.141289,1.084924e+11
3,2012,6.945246e+12,0.164918,1.145396e+12,6.858654e+11,0.133509,9.156928e+10,4.399236e+11,0.173315,7.624551e+10,7.870502e+11,0.141284,1.111974e+11
4,2013,6.998733e+12,0.165286,1.156795e+12,6.925425e+11,0.134138,9.289615e+10,4.558065e+11,0.173743,7.919326e+10,8.064370e+11,0.141500,1.141106e+11


In [5]:
# Tidy for plotting
races = ['White', 'Black', 'Asian', 'Hispanic']
plot_rows = []

for _, r in df_tax.iterrows():
    for race in races:
        val = r.get(f'{race}_TaxContribution', np.nan)
        if pd.notna(val):
            plot_rows.append({'Year': int(r['Year']), 'Race': race, 'TaxContribution': float(val)})

df_plot = pd.DataFrame(plot_rows).sort_values(['Year', 'Race'])
df_plot.head()


,Year,Race,TaxContribution
2,2009,Asian,6.203371e+10
1,2009,Black,8.206165e+10
3,2009,Hispanic,9.633130e+10
0,2009,White,1.059109e+12
6,2010,Asian,6.803557e+10


In [6]:
# Economist-inspired stacked filled line chart
fig = px.area(
    df_plot,
    x='Year',
    y='TaxContribution',
    color='Race',
    color_discrete_map=STYLE['colors'],
    title='Estimated Tax Contribution by Race Over Time (ACS Years Available)'
)

fig.update_traces(mode='lines', line=dict(width=1.2))
fig.update_layout(
    paper_bgcolor=STYLE['background'],
    plot_bgcolor=STYLE['background'],
    font=dict(color=STYLE['font'], family='Georgia'),
    title_font=dict(size=22),
    xaxis=dict(
        title='Year',
        gridcolor=STYLE['grid'],
        zeroline=False
    ),
    yaxis=dict(
        title='Estimated Tax Contribution (USD)',
        gridcolor=STYLE['grid'],
        tickformat='$,.2s',
        zeroline=False
    ),
    legend_title='Race'
)

os.makedirs('output', exist_ok=True)
fig.write_html('output/tax_contribution_by_race_over_time.html')
fig.write_image('output/tax_contribution_by_race_over_time.png', scale=2)
print('Generated output/tax_contribution_by_race_over_time.html')
fig.show()


Generated output/tax_contribution_by_race_over_time.html


In [7]:
# Economist-inspired stacked filled chart: SNAP-based welfare proxy by race over time
WELFARE_VARS = {
    'White':    {'hh_total': 'B22005A_001E', 'hh_assist': 'B22005A_002E', 'median': 'B19013A_001E'},
    'Black':    {'hh_total': 'B22005B_001E', 'hh_assist': 'B22005B_002E', 'median': 'B19013B_001E'},
    'Asian':    {'hh_total': 'B22005D_001E', 'hh_assist': 'B22005D_002E', 'median': 'B19013D_001E'},
    'Hispanic': {'hh_total': 'B22005I_001E', 'hh_assist': 'B22005I_002E', 'median': 'B19013I_001E'},
}


def fetch_welfare_year_data(year):
    fields = ['NAME']
    for race_map in WELFARE_VARS.values():
        fields.extend([race_map['hh_total'], race_map['hh_assist'], race_map['median']])

    df = fetch_with_census_package(year, fields)
    if df is None or df.empty:
        df = fetch_with_rest_api(year, fields, CENSUS_API_KEY)
    if df is None or df.empty:
        return None

    numeric_cols = [f for f in fields if f != 'NAME']
    df = _to_numeric(df, numeric_cols)

    row = {'Year': year}
    for race, vars_map in WELFARE_VARS.items():
        hh_assist = df[vars_map['hh_assist']].sum(skipna=True) if vars_map['hh_assist'] in df.columns else np.nan
        median_income = df[vars_map['median']].mean(skipna=True) if vars_map['median'] in df.columns else np.nan

        # Proxy annual welfare consumption per assisted household with bounded values.
        if pd.notna(median_income):
            annual_benefit = float(np.clip(median_income * 0.18, 2500, 14000))
        else:
            annual_benefit = 6000.0

        row[f'{race}_AssistHH'] = float(hh_assist) if pd.notna(hh_assist) else np.nan
        row[f'{race}_AnnualBenefit'] = annual_benefit
        row[f'{race}_WelfareConsumption'] = row[f'{race}_AssistHH'] * annual_benefit if pd.notna(row[f'{race}_AssistHH']) else np.nan

    return row


welfare_records = []
WELFARE_YEARS = [y for y in YEARS if y <= 2024]
for y in WELFARE_YEARS:
    rec = fetch_welfare_year_data(y)
    if rec is not None:
        welfare_records.append(rec)

if welfare_records:
    df_welfare = pd.DataFrame(welfare_records).sort_values('Year').reset_index(drop=True)
    os.makedirs('output', exist_ok=True)
    df_welfare.to_csv('output/welfare_consumption_by_race_over_time.csv', index=False)
    print('Saved output/welfare_consumption_by_race_over_time.csv')
else:
    fallback_candidates = [
        'output/welfare_consumption_by_race_over_time.csv',
        'welfare_consumption_by_race_over_time.csv',
    ]
    fallback_csv = next((p for p in fallback_candidates if os.path.exists(p)), None)
    if fallback_csv:
        df_welfare = pd.read_csv(fallback_csv)
        print(f'Loaded local fallback: {fallback_csv}')
    else:
        # Keep notebook execution alive when API data is unavailable.
        print('Warning: No welfare data fetched and no local fallback CSV found; skipping welfare chart.')
        df_welfare = pd.DataFrame(columns=['Year'])

welfare_rows = []
races = ['White', 'Black', 'Asian', 'Hispanic']
for _, r in df_welfare.iterrows():
    for race in races:
        val = r.get(f'{race}_WelfareConsumption', np.nan)
        if pd.notna(val):
            welfare_rows.append({'Year': int(r['Year']), 'Race': race, 'WelfareConsumption': float(val)})

df_welfare_plot = pd.DataFrame(welfare_rows)
if not df_welfare_plot.empty:
    df_welfare_plot = df_welfare_plot.sort_values(['Year', 'Race'])

    fig_welfare = px.area(
        df_welfare_plot,
        x='Year',
        y='WelfareConsumption',
        color='Race',
        color_discrete_map=STYLE['colors'],
        title='Estimated SNAP-Linked Welfare Proxy by Race Over Time (ACS Years Available)'
    )

    fig_welfare.update_traces(mode='lines', line=dict(width=1.2))
    fig_welfare.update_layout(
        paper_bgcolor=STYLE['background'],
        plot_bgcolor=STYLE['background'],
        font=dict(color=STYLE['font'], family='Georgia'),
        title_font=dict(size=22),
        xaxis=dict(title='Year', gridcolor=STYLE['grid'], zeroline=False),
        yaxis=dict(title='Estimated SNAP-Linked Welfare Proxy (USD)', gridcolor=STYLE['grid'], tickformat='$,.2s', zeroline=False),
        legend_title='Race'
    )

    fig_welfare.write_html('output/welfare_consumption_by_race_over_time.html')
    fig_welfare.write_image('output/welfare_consumption_by_race_over_time.png', scale=2)
    print('Generated output/welfare_consumption_by_race_over_time.html')
    fig_welfare.show()
else:
    print('No welfare rows available to plot.')


Saved output/welfare_consumption_by_race_over_time.csv
Generated output/welfare_consumption_by_race_over_time.html


## Notes

- This is an **estimate** of tax contribution, not filed tax returns.
- Primary measure is aggregate household income by race from ACS (`B19025*`).
- Fallback estimate is `median income * households` when aggregate income is unavailable for a year.
- Effective tax rate is modeled from income level and bounded between `8%` and `28%`.


## 2024 PUMS Rectangular Proportion Views

These cells use 2024 ACS 1-year PUMS microdata to build disjoint race/ethnicity proportions for population, people in SNAP-recipient households, and an estimated federal income-tax proxy. Tax contribution is modeled from person income; it is not IRS-reported race/ethnicity tax data.


In [8]:
# Shared helper: 2024 ACS PUMS race/ethnicity proportions
import math
import plotly.graph_objects as go

PUMS_YEAR = 2024
PUMS_DATASET = 'acs/acs1/pums'
PUMS_API_BASE = f'https://api.census.gov/data/{PUMS_YEAR}/{PUMS_DATASET}'
PUMS_FIELDS = ['AGEP', 'HISP', 'RAC1P', 'FS', 'PINCP', 'ADJINC', 'PWGTP']
PUMS_CACHE = 'output/pums_race_snap_tax_2024.csv'

PUMS_RACE_ORDER = [
    'Non-Hispanic White',
    'Hispanic or Latino',
    'Non-Hispanic Black',
    'Non-Hispanic Asian',
    'Non-Hispanic Other / Multiracial',
]

PUMS_COLORS = {
    'Non-Hispanic White': STYLE['colors']['White'],
    'Hispanic or Latino': STYLE['colors']['Hispanic'],
    'Non-Hispanic Black': STYLE['colors']['Black'],
    'Non-Hispanic Asian': STYLE['colors']['Asian'],
    'Non-Hispanic Other / Multiracial': '#9c8fb5',
}

STATE_FIPS = [
    '01', '02', '04', '05', '06', '08', '09', '10', '11', '12',
    '13', '15', '16', '17', '18', '19', '20', '21', '22', '23',
    '24', '25', '26', '27', '28', '29', '30', '31', '32', '33',
    '34', '35', '36', '37', '38', '39', '40', '41', '42', '44',
    '45', '46', '47', '48', '49', '50', '51', '53', '54', '55', '56',
]


def fetch_pums_state(state_fips):
    params = {
        'get': ','.join(PUMS_FIELDS),
        'for': 'public use microdata area:*',
        'in': f'state:{state_fips}',
    }
    if CENSUS_API_KEY:
        params['key'] = CENSUS_API_KEY

    r = requests.get(PUMS_API_BASE, params=params, timeout=90)
    r.raise_for_status()
    payload = r.json()
    if not payload or len(payload) < 2:
        return pd.DataFrame(columns=PUMS_FIELDS + ['state'])
    return pd.DataFrame(payload[1:], columns=payload[0])


def classify_pums_race_ethnicity(df):
    hisp = df['HISP'].astype(str).str.zfill(2)
    rac1p = pd.to_numeric(df['RAC1P'], errors='coerce')

    category = np.select(
        [
            hisp.ne('01'),
            rac1p.eq(1),
            rac1p.eq(2),
            rac1p.eq(6),
        ],
        [
            'Hispanic or Latino',
            'Non-Hispanic White',
            'Non-Hispanic Black',
            'Non-Hispanic Asian',
        ],
        default='Non-Hispanic Other / Multiracial',
    )
    return pd.Categorical(category, categories=PUMS_RACE_ORDER, ordered=True)


def estimate_2024_single_federal_income_tax(income):
    income = pd.to_numeric(income, errors='coerce').fillna(0).clip(lower=0)
    taxable = (income - 14600).clip(lower=0)
    brackets = [0, 11600, 47150, 100525, 191950, 243725, 609350]
    rates = [0.10, 0.12, 0.22, 0.24, 0.32, 0.35, 0.37]

    tax = np.zeros(len(taxable), dtype=float)
    for i, lower in enumerate(brackets):
        upper = brackets[i + 1] if i + 1 < len(brackets) else np.inf
        tax += np.clip(taxable - lower, 0, upper - lower) * rates[i]
    return tax


def build_pums_metric_summary(force_refresh=False):
    if os.path.exists(PUMS_CACHE) and not force_refresh:
        df_cached = pd.read_csv(PUMS_CACHE)
        required = {'Metric', 'Category', 'Value', 'Share'}
        if required.issubset(df_cached.columns):
            print(f'Loaded local fallback/cache: {PUMS_CACHE}')
            return df_cached

    totals = {
        'Population': pd.Series(0.0, index=PUMS_RACE_ORDER),
        'SNAP recipient-household population': pd.Series(0.0, index=PUMS_RACE_ORDER),
        'Estimated federal income tax': pd.Series(0.0, index=PUMS_RACE_ORDER),
    }
    failed_states = []

    for state in STATE_FIPS:
        try:
            df_state = fetch_pums_state(state)
        except Exception as exc:
            failed_states.append((state, str(exc)))
            print(f'Warning: failed to fetch state {state}: {exc}')
            continue

        if df_state.empty:
            failed_states.append((state, 'empty response'))
            print(f'Warning: state {state} returned no PUMS rows')
            continue

        for col in ['AGEP', 'RAC1P', 'FS', 'PINCP', 'ADJINC', 'PWGTP']:
            df_state[col] = pd.to_numeric(df_state[col], errors='coerce')

        race_ethnicity = classify_pums_race_ethnicity(df_state)
        weight = df_state['PWGTP'].fillna(0)
        adjinc = df_state['ADJINC'].fillna(1)
        adjinc = np.where(adjinc > 100, adjinc / 1000000, adjinc)
        adjusted_income = df_state['PINCP'].where(df_state['PINCP'] != -19999, 0).fillna(0) * adjinc
        estimated_tax = estimate_2024_single_federal_income_tax(adjusted_income) * weight

        state_metrics = {
            'Population': weight,
            'SNAP recipient-household population': weight.where(df_state['FS'].eq(1), 0),
            'Estimated federal income tax': estimated_tax,
        }
        for metric, values in state_metrics.items():
            grouped = values.groupby(race_ethnicity, observed=False).sum().reindex(PUMS_RACE_ORDER).fillna(0)
            totals[metric] = totals[metric].add(grouped, fill_value=0)

        print(f'Processed state {state}: {len(df_state):,} PUMS person records')

    if failed_states:
        if os.path.exists(PUMS_CACHE):
            print(f'Loaded local fallback/cache because {len(failed_states)} states failed: {PUMS_CACHE}')
            return pd.read_csv(PUMS_CACHE)
        failed_preview = ', '.join(f'{state} ({reason[:60]})' for state, reason in failed_states[:5])
        raise RuntimeError(f'PUMS fetch failed for {len(failed_states)} states and no local fallback CSV exists: {failed_preview}')

    rows = []
    for metric, grouped in totals.items():
        total = grouped.sum()
        for category, value in grouped.items():
            rows.append({
                'Metric': metric,
                'Category': category,
                'Value': float(value),
                'Share': float(value / total) if total else 0,
            })

    df_summary = pd.DataFrame(rows)
    os.makedirs('output', exist_ok=True)
    df_summary.to_csv(PUMS_CACHE, index=False)
    print(f'Saved {PUMS_CACHE}')
    return df_summary

def waffle_counts(shares):
    raw = shares * 100
    counts = np.floor(raw).astype(int)
    remainder = int(100 - counts.sum())
    if remainder > 0:
        bumps = np.argsort(-(raw - counts))[:remainder]
        counts[bumps] += 1
    elif remainder < 0:
        drops = np.argsort(raw - counts)[:abs(remainder)]
        counts[drops] -= 1
    return counts


def make_waffle_dataframe(metric_df):
    metric_df = metric_df.set_index('Category').reindex(PUMS_RACE_ORDER).reset_index()
    counts = waffle_counts(metric_df['Share'].to_numpy())

    points = []
    idx = 0
    for _, row in metric_df.iterrows():
        for _ in range(int(counts[_])):
            points.append({
                'Category': row['Category'],
                'Share': row['Share'],
                'Value': row['Value'],
                'x': idx % 10,
                'y': 9 - (idx // 10),
            })
            idx += 1
    return pd.DataFrame(points)


def plot_rectangular_proportion(df_summary, metric, title, value_label, output_path, value_tickprefix=''):
    metric_df = df_summary[df_summary['Metric'].eq(metric)].copy()
    if metric_df.empty:
        raise ValueError(f'No rows available for metric: {metric}')

    exact_share_sum = metric_df['Share'].sum()
    if not np.isclose(exact_share_sum, 1.0, atol=1e-9):
        raise ValueError(f'{metric} shares sum to {exact_share_sum:.6f}, not 1.0')

    waffle_df = make_waffle_dataframe(metric_df)
    if len(waffle_df) != 100:
        raise ValueError(f'{metric} waffle grid has {len(waffle_df)} points, expected 100')

    fig = go.Figure()
    for category in PUMS_RACE_ORDER:
        d = waffle_df[waffle_df['Category'].eq(category)]
        source_row = metric_df[metric_df['Category'].eq(category)].iloc[0]
        fig.add_trace(go.Scatter(
            x=d['x'],
            y=d['y'],
            mode='markers',
            name=f'{category} ({source_row["Share"]:.1%})',
            marker=dict(size=23, color=PUMS_COLORS[category], line=dict(width=1, color='rgba(47,42,37,0.35)')),
            customdata=np.column_stack([
                np.repeat(source_row['Share'], len(d)),
                np.repeat(source_row['Value'], len(d)),
            ]) if len(d) else None,
            hovertemplate=(
                '<b>%{fullData.name}</b><br>'
                'Share: %{customdata[0]:.2%}<br>'
                f'{value_label}: {value_tickprefix}%{{customdata[1]:,.0f}}'
                '<extra></extra>'
            ),
        ))

    fig.update_layout(
        title=title,
        paper_bgcolor=STYLE['background'],
        plot_bgcolor=STYLE['background'],
        font=dict(color=STYLE['font'], family='Georgia'),
        title_font=dict(size=22),
        width=760,
        height=620,
        xaxis=dict(visible=False, range=[-0.7, 9.7], fixedrange=True),
        yaxis=dict(visible=False, range=[-0.7, 9.7], scaleanchor='x', scaleratio=1, fixedrange=True),
        legend=dict(title='Race / ethnicity', orientation='h', y=-0.08, x=0, font=dict(size=11)),
        margin=dict(l=30, r=30, t=80, b=130),
    )

    os.makedirs('output', exist_ok=True)
    fig.write_html(output_path)
    fig.write_image(output_path.replace('.html', '.png'), scale=2)
    print(f'Generated {output_path}')
    return fig


df_pums_summary = build_pums_metric_summary()
df_pums_summary


Loaded local fallback/cache: output/pums_race_snap_tax_2024.csv


,Metric,Category,Value,Share
0,Population,Non-Hispanic White,1.914133e+08,0.562797
1,Population,Hispanic or Latino,6.799716e+07,0.199926
2,Population,Non-Hispanic Black,3.994157e+07,0.117437
3,Population,Non-Hispanic Asian,2.103497e+07,0.061847
4,Population,Non-Hispanic Other / Multiracial,1.972400e+07,0.057993
5,SNAP recipient-household population,Non-Hispanic White,1.666929e+07,0.353268
6,SNAP recipient-household population,Hispanic or Latino,1.409706e+07,0.298755
7,SNAP recipient-household population,Non-Hispanic Black,1.051237e+07,0.222786
8,SNAP recipient-household population,Non-Hispanic Asian,2.364898e+06,0.050119
9,SNAP recipient-household population,Non-Hispanic Other / Multiracial,3.542398e+06,0.075073


In [9]:
# Rectangular point proportion visualization: race/ethnic population makeup
fig_pums_population = plot_rectangular_proportion(
    df_pums_summary,
    metric='Population',
    title='2024 U.S. Population by Race / Ethnicity (ACS PUMS)',
    value_label='Weighted people',
    output_path='output/population_race_ethnicity_proportion_2024.html',
)
fig_pums_population.show()


Generated output/population_race_ethnicity_proportion_2024.html


In [10]:
# Rectangular point proportion visualization: people in SNAP-recipient households
snap_total = df_pums_summary.loc[df_pums_summary['Metric'].eq('SNAP recipient-household population'), 'Value'].sum()
pop_total = df_pums_summary.loc[df_pums_summary['Metric'].eq('Population'), 'Value'].sum()
if not snap_total < pop_total:
    raise ValueError('SNAP recipient-household population should be lower than total population.')

fig_pums_snap = plot_rectangular_proportion(
    df_pums_summary,
    metric='SNAP recipient-household population',
    title='2024 People in SNAP-Recipient Households by Race / Ethnicity (ACS PUMS)',
    value_label='Weighted people in SNAP-recipient households',
    output_path='output/snap_race_ethnicity_proportion_2024.html',
)
fig_pums_snap.show()


Generated output/snap_race_ethnicity_proportion_2024.html


In [11]:
# Rectangular point proportion visualization: estimated tax contribution makeup
fig_pums_tax = plot_rectangular_proportion(
    df_pums_summary,
    metric='Estimated federal income tax',
    title='2024 Estimated Federal Income Tax by Race / Ethnicity (ACS PUMS Income Proxy)',
    value_label='Estimated federal tax',
    output_path='output/tax_race_ethnicity_proportion_2024.html',
    value_tickprefix='$',
)
fig_pums_tax.show()


Generated output/tax_race_ethnicity_proportion_2024.html
